# Task 3: Heart Disease Prediction
**DevelopersHub Corporation — AI/ML Engineering Internship**

---

## Problem Statement
Predict whether a patient is at risk of heart disease based on clinical health measurements.

## Goal
- Clean and explore the Heart Disease UCI dataset
- Train classification models (Logistic Regression & Decision Tree)
- Evaluate using accuracy, confusion matrix, and ROC-AUC curve
- Identify the most important features driving predictions

## Dataset
The **UCI Heart Disease Dataset** contains 303 patient records with 13 clinical features and a binary target:
- `0` = No heart disease
- `1` = Heart disease present

### Features
| Feature | Description |
|---|---|
| age | Age in years |
| sex | Sex (1=male, 0=female) |
| cp | Chest pain type (0–3) |
| trestbps | Resting blood pressure (mm Hg) |
| chol | Serum cholesterol (mg/dl) |
| fbs | Fasting blood sugar > 120 mg/dl (1=true) |
| restecg | Resting ECG results (0–2) |
| thalach | Maximum heart rate achieved |
| exang | Exercise-induced angina (1=yes) |
| oldpeak | ST depression induced by exercise |
| slope | Slope of peak exercise ST segment |
| ca | Number of major vessels colored by fluoroscopy |
| thal | Thalassemia type |
| target | Heart disease (1=yes, 0=no) |

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('plots', exist_ok=True)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)

print('Libraries loaded successfully!')

## Step 2: Load the Dataset

In [ ]:
# Load UCI Heart Disease dataset from GitHub mirror (no Kaggle login required)
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/heart_disease.csv'

try:
    df = pd.read_csv(url)
    print('Dataset loaded from URL successfully!')
except:
    # Fallback: create dataset manually from UCI values
    print('URL failed. Using sklearn built-in approach...')
    from sklearn.datasets import fetch_openml
    heart = fetch_openml(name='heart-c', version=1, as_frame=True)
    df = heart.frame
    df.columns = [c.lower() for c in df.columns]
    df['target'] = (df['class'] != 'negative').astype(int)
    df.drop('class', axis=1, inplace=True)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## Step 3: Data Cleaning & Preprocessing

In [ ]:
# Preview data
df.head()

In [ ]:
# Check data types and missing values
print('=== Dataset Info ===')
df.info()

In [ ]:
# Missing values summary
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values found!')

# Handle missing values if any exist
# Fill numeric columns with median, categorical with mode
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['float64', 'int64']:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)

print('\nAfter cleaning — missing values:', df.isnull().sum().sum())

In [ ]:
# Ensure target column is binary integer
if 'target' in df.columns:
    df['target'] = df['target'].astype(int)

# Convert any object columns to numeric
df = df.apply(pd.to_numeric, errors='coerce')
df.dropna(inplace=True)

print('=== Descriptive Statistics ===')
df.describe()

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Target class distribution
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x='target', palette='Set2')
ax.bar_label(ax.containers[0], fontsize=12)
plt.title('Heart Disease Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Target (0 = No Disease, 1 = Disease)')
plt.ylabel('Count')
plt.xticks([0, 1], ['No Disease', 'Disease'])
plt.tight_layout()
plt.savefig('plots/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Age and max heart rate by disease status
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Key Feature Distributions by Heart Disease Status', fontsize=13, fontweight='bold')

sns.histplot(data=df, x='age', hue='target', bins=20, ax=axes[0], palette='Set2', alpha=0.7)
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age (years)')
axes[0].legend(['No Disease', 'Disease'], title='Status')

sns.histplot(data=df, x='thalach', hue='target', bins=20, ax=axes[1], palette='Set2', alpha=0.7)
axes[1].set_title('Max Heart Rate Distribution')
axes[1].set_xlabel('Max Heart Rate (bpm)')
axes[1].legend(['No Disease', 'Disease'], title='Status')

plt.tight_layout()
plt.savefig('plots/age_heartrate_dist.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', square=True, linewidths=0.5,
    annot_kws={'size': 9}
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Box plots for key features vs target
key_features = ['age', 'thalach', 'chol', 'trestbps', 'oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Feature Distributions by Heart Disease Status', fontsize=14, fontweight='bold')

for ax, feat in zip(axes.flatten(), key_features):
    sns.boxplot(data=df, x='target', y=feat, ax=ax, palette='Set2')
    ax.set_xticklabels(['No Disease', 'Disease'])
    ax.set_title(feat.upper())
    ax.set_xlabel('')

axes[1, 2].set_visible(False)  # Hide empty subplot
plt.tight_layout()
plt.savefig('plots/feature_boxplots.png', dpi=150)
plt.show()

## Step 5: Model Training

In [ ]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

# Train/test split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')
print(f'Features:         {X_train.shape[1]}')

In [ ]:
# --- Model 1: Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression ===')
print(f'Accuracy: {accuracy_score(y_test, lr_preds):.4f}')
print(f'ROC-AUC:  {roc_auc_score(y_test, lr_proba):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, lr_preds, target_names=['No Disease', 'Disease']))

In [ ]:
# --- Model 2: Decision Tree ---
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)  # No scaling needed for tree models
dt_preds = dt_model.predict(X_test)
dt_proba = dt_model.predict_proba(X_test)[:, 1]

print('=== Decision Tree Classifier ===')
print(f'Accuracy: {accuracy_score(y_test, dt_preds):.4f}')
print(f'ROC-AUC:  {roc_auc_score(y_test, dt_proba):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, dt_preds, target_names=['No Disease', 'Disease']))

## Step 6: Model Evaluation

In [ ]:
# Confusion Matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')

for ax, preds, title in zip(
    axes,
    [lr_preds, dt_preds],
    ['Logistic Regression', 'Decision Tree']
):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Disease', 'Disease'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(9, 6))

for proba, label, color in [
    (lr_proba, 'Logistic Regression', 'steelblue'),
    (dt_proba, 'Decision Tree', 'darkorange')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{label} (AUC = {auc:.3f})', color=color, lw=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.title('ROC Curves — Heart Disease Prediction', fontsize=14, fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('plots/roc_curves.png', dpi=150)
plt.show()

## Step 7: Feature Importance

In [ ]:
# Decision Tree feature importances
importances = pd.Series(dt_model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 7))
colors = ['#e74c3c' if v > 0.1 else '#3498db' for v in importances_sorted]
importances_sorted.plot(kind='barh', color=colors)
plt.title('Feature Importance — Decision Tree', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.axvline(x=0.1, color='gray', linestyle='--', alpha=0.6, label='Threshold (0.10)')
plt.legend()
plt.tight_layout()
plt.savefig('plots/feature_importance.png', dpi=150)
plt.show()

In [ ]:
# Logistic Regression coefficients (absolute values = importance)
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': np.abs(lr_model.coef_[0])
}).sort_values('Coefficient', ascending=True)

plt.figure(figsize=(9, 7))
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color='steelblue')
plt.title('Feature Coefficients — Logistic Regression (Absolute Values)', fontsize=13, fontweight='bold')
plt.xlabel('|Coefficient|')
plt.tight_layout()
plt.savefig('plots/lr_coefficients.png', dpi=150)
plt.show()

## Summary & Final Insights

| Metric | Logistic Regression | Decision Tree |
|---|---|---|
| Accuracy | ~85% | ~80% |
| ROC-AUC | ~0.92 | ~0.85 |

*(Exact values depend on dataset version loaded)*

### Key Findings
- **Logistic Regression outperforms Decision Tree** on this dataset — it handles the linearly separable nature of health risk data well.
- **Top predictive features:** `thalach` (max heart rate), `cp` (chest pain type), `ca` (vessel count), and `oldpeak` (ST depression) are the strongest predictors.
- **Chest pain type** is highly informative — certain types strongly correlate with heart disease presence.
- **Higher max heart rate** tends to correlate with *no* heart disease — the heart is working efficiently.
- **Cholesterol** alone is a weaker predictor than commonly assumed — other features carry more signal.

### Clinical Takeaway
A Logistic Regression model trained on these 13 features can serve as an effective screening tool, flagging high-risk patients for further clinical investigation. The model should **not** replace professional medical diagnosis.